# Intermediate Topics in LangGraph

## Tutorials Covered:
1. Multi-Agent Systems
2. Human-in-the-Loop
3. Streaming Responses
4. State Persistence

## 1. Multi-Agent Systems

Learning objectives:
- Coordinate multiple agents in a single graph
- Understand agent communication patterns
- Implement collaborative agent behaviors

In [ ]:
# Tutorial 1: Multi-Agent Systems

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json

print("=== Multi-Agent Systems in LangGraph ===\n")

# Define state for multi-agent system
class MultiAgentState(TypedDict):
    task: str
    agents: dict  # Dictionary to store agent-specific data
    agent_responses: List[dict]
    consensus: str
    current_agent: str
    iteration: int

# Define different agents

def researcher_agent(state):
    print(f"Researcher agent analyzing task: {state['task']}")
    
    # Simulate research activity
    research_result = f"Research findings on '{state['task']}': Based on analysis, key factors include A, B, and C. Further investigation suggests approach X might be optimal."
    
    agent_response = {
        "agent": "researcher",
        "response": research_result,
        "confidence": 0.8
    }
    
    return {
        "agent_responses": state['agent_responses'] + [agent_response],
        "agents": {**state['agents'], "researcher": research_result},
        "current_agent": "analyst"
    }

def analyst_agent(state):
    print(f"Analyst agent reviewing research for task: {state['task']}")
    
    # Analyze the research
    analysis_result = f"Analysis of research: The data indicates strong correlation between variables. Risk assessment shows moderate risk level. Recommendation: proceed with caution and monitor KPIs D and E."
    
    agent_response = {
        "agent": "analyst",
        "response": analysis_result,
        "confidence": 0.75
    }
    
    return {
        "agent_responses": state['agent_responses'] + [agent_response],
        "agents": {**state['agents'], "analyst": analysis_result},
        "current_agent": "writer"
    }

def writer_agent(state):
    print(f"Writer agent synthesizing information for task: {state['task']}")
    
    # Synthesize the information from other agents
    synthesis = f"Synthesized report: Based on research and analysis, for task '{state['task']}', we recommend proceeding with approach X while monitoring for risks related to factors D and E. Implementation should follow best practices for A, B, and C."
    
    agent_response = {
        "agent": "writer",
        "response": synthesis,
        "confidence": 0.9
    }
    
    return {
        "agent_responses": state['agent_responses'] + [agent_response],
        "agents": {**state['agents'], "writer": synthesis},
        "current_agent": "reviewer"
    }

def reviewer_agent(state):
    print(f"Reviewer agent evaluating solution for task: {state['task']}")
    
    # Review the synthesized solution
    review_result = f"Review: The proposed solution addresses the core requirements. Minor suggestions include clarifying approach X and adding contingency plans for identified risks. Overall confidence: high."
    
    agent_response = {
        "agent": "reviewer",
        "response": review_result,
        "confidence": 0.85
    }
    
    # Form consensus based on all agent responses
    all_responses = state['agent_responses'] + [agent_response]
    consensus = f"Consensus achieved after review by all agents: For task '{state['task']}', implement approach X with risk monitoring and contingency planning. Supported by Researcher, Analyst, Writer, and Reviewer agents."
    
    return {
        "agent_responses": all_responses,
        "agents": {**state['agents'], "reviewer": review_result},
        "consensus": consensus,
        "current_agent": "none"
    }

# Routing function to determine which agent should act next
def agent_router(state):
    print(f"Routing to next agent: {state['current_agent']}")
    return state['current_agent']

print("1. MULTI-AGENT SYSTEMS coordinate multiple specialized agents")
print("   Each agent performs a specific role in solving complex tasks\n")

# Create the multi-agent graph
builder = StateGraph(MultiAgentState)

# Add nodes for each agent
builder.add_node("researcher", researcher_agent)
builder.add_node("analyst", analyst_agent)
builder.add_node("writer", writer_agent)
builder.add_node("reviewer", reviewer_agent)

# Set up the flow
builder.add_edge("__start__", "researcher")

# Add conditional edges based on current_agent
builder.add_conditional_edges(
    "researcher",
    agent_router,
    {
        "analyst": "analyst",
        "writer": "writer",
        "reviewer": "reviewer"
    }
)

builder.add_conditional_edges(
    "analyst",
    agent_router,
    {
        "writer": "writer",
        "reviewer": "reviewer"
    }
)

builder.add_conditional_edges(
    "writer",
    agent_router,
    {
        "reviewer": "reviewer"
    }
)

builder.add_edge("reviewer", "__end__")

# Compile the multi-agent system
multi_agent_system = builder.compile()

print("2. Executing the multi-agent system with a complex task:\n")

# Test the multi-agent system with a complex task
complex_task = "Design an effective customer retention strategy for an e-commerce platform"

result = multi_agent_system.invoke({
    "task": complex_task,
    "agents": {},
    "agent_responses": [],
    "consensus": "",
    "current_agent": "",
    "iteration": 0
})

print(f"Final consensus: {result['consensus']}")
print(f"Number of agent responses: {len(result['agent_responses'])}")
print(f"Agents involved: {list(result['agents'].keys())}")

print("\n3. Key concepts of multi-agent systems:")
print("   - Different agents specialize in different aspects of a problem")
print("   - Information flows between agents to build a comprehensive solution")
print("   - Consensus emerges from the collaboration of multiple perspectives")
print("   - Each agent contributes its expertise to the final outcome")

## 2. Human-in-the-Loop

Learning objectives:
- Add human feedback points in automated workflows
- Implement approval mechanisms
- Design interactive human-agent collaboration

In [ ]:
# Tutorial 2: Human-in-the-Loop

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import time

print("=== Human-in-the-Loop in LangGraph ===\n")

# Define state for human-in-the-loop system
class HumanInLoopState(TypedDict):
    task_description: str
    ai_proposal: str
    human_feedback: str
    human_approval: bool
    iteration_count: int
    decision_log: List[str]
    requires_human_input: bool

# Define nodes for human-in-the-loop system

def ai_analyst_node(state):
    print(f"AI Analyst generating proposal for: {state['task_description']}")
    
    # Generate AI proposal based on task
    proposal = f"AI Proposal: To address '{state['task_description']}', I recommend implementing solution Y which involves steps 1, 2, and 3. This approach has estimated success probability of 85%."
    
    return {
        "ai_proposal": proposal,
        "decision_log": state['decision_log'] + [f"AI generated proposal: {proposal[:50]}..."]
    }

def human_review_node(state):
    print(f"Requesting human review for proposal: {state['ai_proposal'][:60]}...")
    
    # In a real implementation, this would interface with a human reviewer
    # For simulation, we'll mimic human input
    print("   Human reviewer is reviewing the proposal...")
    time.sleep(0.5)  # Simulate review time
    
    # Simulate human feedback - sometimes approve, sometimes request changes
    import random
    if random.random() < 0.6:  # 60% approval rate
        human_feedback = "APPROVED: The proposal looks good and addresses the requirements."
        human_approval = True
    else:
        human_feedback = "FEEDBACK: Consider adding more detail about step 2 and addressing potential risks."
        human_approval = False
    
    return {
        "human_feedback": human_feedback,
        "human_approval": human_approval,
        "decision_log": state['decision_log'] + [f"Human review: {human_feedback[:30]}..."]
    }

def ai_revision_node(state):
    print(f"AI revising proposal based on feedback: {state['human_feedback'][:50]}...")
    
    # Revise proposal based on human feedback
    revision = f"REVISION: Based on feedback '{state['human_feedback']}', the revised proposal now includes additional details about step 2 and risk mitigation strategies. The solution now involves steps 1, 2 (enhanced), 3, and 4 (risk mitigation)."
    
    new_iteration = state['iteration_count'] + 1
    
    # Check if we should continue the loop
    requires_input = new_iteration < 3 and not state['human_approval']  # Max 3 iterations
    
    return {
        "ai_proposal": revision,
        "iteration_count": new_iteration,
        "requires_human_input": requires_input,
        "decision_log": state['decision_log'] + [f"AI revision iteration {new_iteration}"]
    }

def final_decision_node(state):
    print(f"Making final decision. Approved: {state['human_approval']}")
    
    if state['human_approval']:
        final_decision = f"FINAL DECISION: Proposal approved by human reviewer. Ready for implementation. Summary: {state['ai_proposal'][:100]}..."
    else:
        final_decision = f"FINAL DECISION: Proposal required too many revisions. Paused for further human evaluation. Current status: {state['ai_proposal'][:100]}..."
    
    return {
        "decision_log": state['decision_log'] + ["Final decision recorded"]
    }

# Routing function to determine next action
def human_loop_router(state):
    print(f"Routing based on approval status: {state['human_approval']}, Iteration: {state['iteration_count']}")
    
    if state['human_approval']:
        return 'final_decision'
    elif state['iteration_count'] < 3:
        return 'ai_revision'
    else:
        return 'final_decision'

print("1. HUMAN-IN-THE-LOOP integrates human judgment into automated workflows")
print("   Humans can approve, reject, or provide feedback to AI-generated outputs\n")

# Create the human-in-the-loop graph
builder = StateGraph(HumanInLoopState)

# Add nodes
builder.add_node("ai_analyst", ai_analyst_node)
builder.add_node("human_review", human_review_node)
builder.add_node("ai_revision", ai_revision_node)
builder.add_node("final_decision", final_decision_node)

# Set up the flow
builder.add_edge("__start__", "ai_analyst")
builder.add_edge("ai_analyst", "human_review")

# Add conditional edges for the loop
builder.add_conditional_edges(
    "human_review",
    human_loop_router,
    {
        "ai_revision": "ai_revision",
        "final_decision": "final_decision"
    }
)

builder.add_conditional_edges(
    "ai_revision",
    human_loop_router,
    {
        "human_review": "human_review",  # Loop back to human review
        "final_decision": "final_decision"
    }
)

builder.add_edge("final_decision", "__end__")

# Compile the human-in-the-loop system
human_in_loop_system = builder.compile()

print("2. Executing the human-in-the-loop system with different tasks:\n")

# Test the human-in-the-loop system with different tasks
tasks = [
    "Improve customer support response times",
    "Optimize the checkout process for mobile users",
    "Implement a new recommendation algorithm"
]

for i, task in enumerate(tasks):
    print(f"--- Task {i+1}: {task} ---")
    result = human_in_loop_system.invoke({
        "task_description": task,
        "ai_proposal": "",
        "human_feedback": "",
        "human_approval": False,
        "iteration_count": 0,
        "decision_log": [f"Started processing task {i+1}"],
        "requires_human_input": True
    })
    print(f"Final approval status: {result['human_approval']}")
    print(f"Iterations completed: {result['iteration_count']}")
    print(f"Decision log: {result['decision_log'][-2:]}")
    print()

print("3. Key concepts of human-in-the-loop systems:")
print("   - Humans provide critical oversight for AI-generated proposals")
print("   - Feedback loops allow iterative improvement")
print("   - Approval mechanisms ensure quality control")
print("   - Systems can balance automation with human judgment")

## 3. Streaming Responses

Learning objectives:
- Implement real-time response streaming
- Handle incremental updates
- Manage streaming data flows

In [ ]:
# Tutorial 3: Streaming Responses

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import time
import asyncio

print("=== Streaming Responses in LangGraph ===\n")

# Define state for streaming responses
class StreamingState(TypedDict):
    query: str
    streaming_response: str
    chunks_received: List[str]
    processing_status: str
    stream_complete: bool
    tokens_per_second: float

# Define nodes for streaming response system

def query_parser_node(state):
    print(f"Parsing query for streaming: {state['query']}")
    
    # Parse the query and prepare for streaming
    parsed_info = f"Parsed query: {state['query']}. Preparing streaming response..."
    
    return {
        "chunks_received": [parsed_info],
        "processing_status": "parsing_complete",
        "streaming_response": parsed_info
    }

def stream_generator_node(state):
    print(f"Generating streaming response for: {state['query']}")
    
    # Simulate streaming response generation
    query_lower = state['query'].lower()
    
    if "weather" in query_lower:
        chunks = [
            "Weather Report: ",
            "Today's forecast shows ",
            "partly cloudy conditions with ",
            "scattered showers possible in the ",
            "afternoon. Temperature range is ",
            "between 18-24°C. ",
            "UV index is moderate. ",
            "Recommended to carry an umbrella. ",
            "End of weather report."
        ]
    elif "news" in query_lower:
        chunks = [
            "Latest News: ",
            "Global markets showed mixed ",
            "performance today with tech ",
            "stocks leading gains. ",
            "Renewable energy sector ",
            "announced major breakthrough. ",
            "Scientists discovered new ",
            "approach to carbon capture. ",
            "End of news update."
        ]
    else:
        chunks = [
            "Analysis Result: ",
            "Based on the query, ",
            "comprehensive analysis shows ",
            "several key factors at play. ",
            "Primary considerations include ",
            "market trends, user behavior, ",
            "and technological impacts. ",
            "Recommendation is to proceed ",
            "with strategic implementation. ",
            "End of analysis."
        ]
    
    # Simulate streaming by adding chunks gradually
    all_chunks = state['chunks_received'] + chunks
    full_response = " ".join(all_chunks)
    
    return {
        "chunks_received": all_chunks,
        "streaming_response": full_response,
        "processing_status": "generating_stream",
        "tokens_per_second": len(full_response.split()) / 5.0  # Simulated rate
    }

def stream_aggregator_node(state):
    print(f"Aggregating streaming response")
    
    # Aggregate all received chunks into a cohesive response
    aggregated = " ".join(state['chunks_received'])
    
    # Add conclusion
    conclusion = " [STREAMING COMPLETE]"
    final_response = aggregated + conclusion
    
    return {
        "streaming_response": final_response,
        "processing_status": "aggregation_complete",
        "stream_complete": True
    }

def progress_monitor_node(state):
    print(f"Monitoring stream progress: {len(state['chunks_received'])} chunks received")
    
    # Calculate metrics
    total_length = len(state['streaming_response'])
    progress_percentage = min(100, len(" ".join(state['chunks_received'])) * 100 // (total_length or 1))
    
    return {
        "processing_status": f"progress_{progress_percentage}%",
        "chunks_received": state['chunks_received']
    }

print("1. STREAMING RESPONSES provide real-time, incremental output")
print("   Rather than waiting for complete processing, users receive updates as they're generated\n")

# Create the streaming response graph
builder = StateGraph(StreamingState)

# Add nodes
builder.add_node("query_parser", query_parser_node)
builder.add_node("stream_generator", stream_generator_node)
builder.add_node("progress_monitor", progress_monitor_node)
builder.add_node("stream_aggregator", stream_aggregator_node)

# Set up the flow
builder.add_edge("__start__", "query_parser")
builder.add_edge("query_parser", "stream_generator")
builder.add_edge("stream_generator", "progress_monitor")
builder.add_edge("progress_monitor", "stream_aggregator")
builder.add_edge("stream_aggregator", "__end__")

# Compile the streaming system
streaming_system = builder.compile()

print("2. Executing the streaming response system with different queries:\n")

# Test the streaming system with different queries
test_queries = [
    "What is the weather forecast for tomorrow?",
    "Give me the latest news updates.",
    "Analyze the market trends for tech stocks."
]

for i, query in enumerate(test_queries):
    print(f"--- Query {i+1}: {query} ---")
    result = streaming_system.invoke({
        "query": query,
        "streaming_response": "",
        "chunks_received": [],
        "processing_status": "initial",
        "stream_complete": False,
        "tokens_per_second": 0.0
    })
    print(f"Final response length: {len(result['streaming_response'])} chars")
    print(f"Chunks received: {len(result['chunks_received'])}")
    print(f"Processing status: {result['processing_status']}")
    print(f"Tokens per second: {result['tokens_per_second']:.2f}")
    print(f"Stream complete: {result['stream_complete']}")
    print()

print("3. Key concepts of streaming responses:")
print("   - Users receive incremental updates instead of waiting for completion")
print("   - Progress can be monitored and displayed")
print("   - Large responses can be handled efficiently")
print("   - Real-time feedback improves user experience")

## 4. State Persistence

Learning objectives:
- Save and restore graph state across sessions
- Implement persistence strategies
- Manage state lifecycle

In [ ]:
# Tutorial 4: State Persistence

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json
import os
import time

print("=== State Persistence in LangGraph ===\n")

# Define state for persistence system
class PersistenceState(TypedDict):
    task_id: str
    task_description: str
    current_step: str
    step_results: List[dict]
    persisted_state_path: str
    execution_history: List[str]
    checkpoint_time: float

# Simulated storage backend

class StateStorage:
    @staticmethod
    def save_state(task_id: str, state: dict) -> str:
        # Create a file-based storage simulation
        filename = f"/tmp/langgraph_state_{task_id}.json"
        with open(filename, 'w') as f:
            json.dump(state, f, indent=2, default=str)
        return filename
    
    @staticmethod
    def load_state(task_id: str) -> dict:
        filename = f"/tmp/langgraph_state_{task_id}.json"
        if os.path.exists(filename):
            with open(filename, 'r') as f:
                return json.load(f)
        return {}
    
    @staticmethod
    def delete_state(task_id: str):
        filename = f"/tmp/langgraph_state_{task_id}.json"
        if os.path.exists(filename):
            os.remove(filename)

# Define nodes for persistence system

def task_initializer_node(state):
    print(f"Initializing task: {state['task_description']}")
    
    # Initialize the task
    step_result = {
        "step": "initialization",
        "status": "completed",
        "timestamp": time.time()
    }
    
    # Save initial state
    persisted_path = StateStorage.save_state(state['task_id'], state)
    
    return {
        "current_step": "initialization",
        "step_results": [step_result],
        "persisted_state_path": persisted_path,
        "execution_history": [f"Task {state['task_id']} initialized at {time.time()}"]
    }

def data_processing_node(state):
    print(f"Processing data for task: {state['task_id']}")
    
    # Simulate data processing
    time.sleep(0.2)  # Simulate processing time
    
    step_result = {
        "step": "data_processing",
        "status": "completed",
        "data_processed": 150,
        "timestamp": time.time()
    }
    
    # Load existing state to append results
    current_state = StateStorage.load_state(state['task_id'])
    current_results = current_state.get('step_results', [])
    
    # Update and save state
    updated_results = current_results + [step_result]
    updated_state = {**state, "step_results": updated_results}
    persisted_path = StateStorage.save_state(state['task_id'], updated_state)
    
    return {
        "current_step": "data_processing",
        "step_results": updated_results,
        "persisted_state_path": persisted_path,
        "execution_history": state['execution_history'] + [f"Data processing completed at {time.time()}"]
    }

def analysis_node(state):
    print(f"Analyzing results for task: {state['task_id']}")
    
    # Simulate analysis
    time.sleep(0.2)  # Simulate analysis time
    
    step_result = {
        "step": "analysis",
        "status": "completed",
        "insights": ["Key finding 1", "Key finding 2", "Key finding 3"],
        "timestamp": time.time()
    }
    
    # Update and save state
    current_results = state['step_results'] + [step_result]
    persisted_path = StateStorage.save_state(state['task_id'], {
        **state,
        "step_results": current_results
    })
    
    return {
        "current_step": "analysis",
        "step_results": current_results,
        "persisted_state_path": persisted_path,
        "execution_history": state['execution_history'] + [f"Analysis completed at {time.time()}"]
    }

def report_generation_node(state):
    print(f"Generating report for task: {state['task_id']}")
    
    # Simulate report generation
    time.sleep(0.2)  # Simulate report generation time
    
    step_result = {
        "step": "report_generation",
        "status": "completed",
        "report_url": f"/reports/task_{state['task_id']}_report.pdf",
        "timestamp": time.time()
    }
    
    # Update and save state
    current_results = state['step_results'] + [step_result]
    persisted_path = StateStorage.save_state(state['task_id'], {
        **state,
        "step_results": current_results
    })
    
    return {
        "current_step": "report_generation",
        "step_results": current_results,
        "persisted_state_path": persisted_path,
        "execution_history": state['execution_history'] + [f"Report generation completed at {time.time()}"]
    }

def state_restoration_node(state):
    print(f"Restoring state for task: {state['task_id']} from {state['persisted_state_path']}")
    
    # Load state from storage
    restored_state = StateStorage.load_state(state['task_id'])
    
    # Add restoration info
    restoration_info = {
        "step": "state_restoration",
        "status": "completed",
        "restored_from": state['persisted_state_path'],
        "timestamp": time.time()
    }
    
    # Update and save state
    current_results = state['step_results'] + [restoration_info]
    persisted_path = StateStorage.save_state(state['task_id'], {
        **state,
        "step_results": current_results
    })
    
    return {
        "step_results": current_results,
        "execution_history": state['execution_history'] + [f"State restoration completed at {time.time()}"]
    }

print("1. STATE PERSISTENCE saves and restores graph state across sessions")
print("   This allows workflows to resume from where they left off\n")

# Create the persistence system graph
builder = StateGraph(PersistenceState)

# Add nodes
builder.add_node("task_initializer", task_initializer_node)
builder.add_node("data_processing", data_processing_node)
builder.add_node("analysis", analysis_node)
builder.add_node("report_generation", report_generation_node)
builder.add_node("state_restoration", state_restoration_node)

# Set up the flow
builder.add_edge("__start__", "task_initializer")
builder.add_edge("task_initializer", "data_processing")
builder.add_edge("data_processing", "analysis")
builder.add_edge("analysis", "report_generation")
builder.add_edge("report_generation", "state_restoration")
builder.add_edge("state_restoration", "__end__")

# Compile the persistence system
persistence_system = builder.compile()

print("2. Executing the state persistence system with different tasks:\n")

# Test the persistence system with different tasks
tasks = [
    {"task_id": "task_001", "task_description": "Process quarterly sales data"},
    {"task_id": "task_002", "task_description": "Analyze customer feedback survey"},
    {"task_id": "task_003", "task_description": "Generate marketing campaign report"}
]

for i, task_params in enumerate(tasks):
    print(f"--- Task {i+1}: {task_params['task_description']} ---")
    
    # Initialize the state for this task
    initial_state = {
        "task_id": task_params['task_id'],
        "task_description": task_params['task_description'],
        "current_step": "",
        "step_results": [],
        "persisted_state_path": "",
        "execution_history": [f"Task {task_params['task_id']} started"],
        "checkpoint_time": time.time()
    }
    
    result = persistence_system.invoke(initial_state)
    
    print(f"Task ID: {result['task_id']}")
    print(f"Current step: {result['current_step']}")
    print(f"Step results: {len(result['step_results'])} steps completed")
    print(f"Execution history: {len(result['execution_history'])} events")
    print(f"Persisted at: {result['persisted_state_path']}")
    
    # Verify persistence worked by loading the state
    loaded_state = StateStorage.load_state(task_params['task_id'])
    print(f"State verified: {bool(loaded_state)}")
    print()

# Cleanup temporary files
for task_params in tasks:
    try:
        StateStorage.delete_state(task_params['task_id'])
    except FileNotFoundError:
        pass

print("3. Key concepts of state persistence:")
print("   - State can be saved to external storage during execution")
print("   - Workflows can resume from saved checkpoints")
print("   - Persistence enables fault tolerance")
print("   - Historical state can be audited and analyzed")